## Count AbacusSummit halos (CompaSO `halo_info`)

This notebook counts the total number of halos in an AbacusSummit `halo_info` directory.

Target directory (from user):

- `/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info`


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import re
import sys
import numpy as np

HALO_INFO_DIR = Path(
    os.environ.get(
        "ABACUS_HALO_INFO_DIR",
        "/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info",
    )
).expanduser()

HALO_INFO_DIR

In [ ]:
from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog

In [ ]:
halos = CompaSOHaloCatalog('/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info/halo_info_000.asdf', cleaned=False)

In [ ]:
import gc
import numpy as np
from pathlib import Path
from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog

HALO_INFO_DIR = Path("/pscratch/sd/d/dkololgi/AbacusSummit_densities/AbacusSummit_base_c000_ph000/halos/z0.200/halo_info")
M_MIN = 1e12  # Msun/h

x_keep = []
N_keep = []

files = sorted(HALO_INFO_DIR.glob("halo_info_*.asdf"))
if not files:
    raise FileNotFoundError(f"No halo_info_*.asdf in {HALO_INFO_DIR}")

pm = None  # particle mass is the same for all files in a snapshot

for i, fp in enumerate(files):
    # Only load needed halo fields.
    # (API differs across abacusnbody versions; handle both patterns.)
    
    cat = CompaSOHaloCatalog(str(fp), cleaned=False, fields=["N", "x_com"])

    if pm is None:
        pm = float(cat.header["ParticleMassHMsun"])
        N_min = int(np.ceil(M_MIN / pm))

    halos = cat.halos
    N = halos["N"]
    m = N >= N_min
    if np.any(m):
        x_keep.append(halos["x_com"][m].astype(np.float32, copy=False))
        N_keep.append(N[m].astype(np.int32, copy=False))

    # Free everything tied to this file before moving on.
    del halos, N, m, cat
    if (i + 1) % 2 == 0:
        gc.collect()

x_com = np.concatenate(x_keep, axis=0) if x_keep else np.empty((0, 3), dtype=np.float32)
N = np.concatenate(N_keep, axis=0) if N_keep else np.empty((0,), dtype=np.int32)
M = N.astype(np.float64) * pm if pm is not None else np.empty((0,), dtype=np.float64)

print("Selected halos:", len(N))
print("ParticleMassHMsun:", pm)
print("Min mass (Msun/h):", M.min() if len(M) else None)

In [ ]:
def count_halos_with_compaso_catalog(halo_info_dir: Path) -> tuple[int, list[tuple[str, int]]]:
    """Count halos by reading CompaSO halo_info via Abacus Python tooling.

    Returns:
      total_halos, per_file_counts (filename, n_halos)
    """
    # Preferred import path for the AbacusSummit/abacusutils stack.
    from abacusnbody.data.compaso_halo_catalog import CompaSOHaloCatalog

    halo_info_dir = halo_info_dir.resolve()
    files = sorted(halo_info_dir.glob("halo_info_*.asdf"))
    if not files:
        raise FileNotFoundError(f"No halo_info_*.asdf files found in {halo_info_dir}")

    # CompaSOHaloCatalog expects the directory containing halo_info_*.asdf.
    # NOTE: some abacusnbody versions reject a `load=...` kwarg (your error).
    # We only need the halo table length, so we use the simplest constructor.
    try:
        cat = CompaSOHaloCatalog(str(halo_info_dir), cleaned=False)
    except TypeError:
        # Fallback for signatures that don't accept `cleaned`.
        cat = CompaSOHaloCatalog(str(halo_info_dir))

    n = len(cat.halos)
    return n, [("<directory>", n)]


try:
    import abacusnbody  # noqa: F401
    total_halos, per_file = count_halos_with_compaso_catalog(HALO_INFO_DIR)
    print(f"abacusnbody total halos: {total_halos:,}")
    if per_file and per_file[0][0] != "<directory>":
        for name, n in per_file:
            print(f"  {name}: {n:,}")
except ModuleNotFoundError as e:
    print("abacusnbody not available in this kernel. Use Option B.")
    print("Import error:", e)

### Option B: read ASDF after installing the right codec support

Your current base Python environment may not support the Abacus ASDF compression used in `halo_info_*.asdf`.

Two common ways to fix this:

1. Install `abacusutils` into the kernel environment (recommended).
2. Install ASDF plugins that support the compression codec in your files.

Once your environment can open the ASDF, the code below counts halos by finding a table-like array and using its first dimension.

In [ ]:
def _pick_halo_table_len(tree) -> int:
    """Heuristically find the halo count from an ASDF tree.

    We look for a dict-like group named 'halos' first, otherwise search for
    a 1D array field with a plausible halo length.
    """
    # Preferred: tree['halos'] is often a structured array or dict of arrays.
    if isinstance(tree, dict) and "halos" in tree:
        halos = tree["halos"]
        if hasattr(halos, "__len__") and not isinstance(halos, dict):
            return len(halos)
        if isinstance(halos, dict):
            # choose first array-like field
            for v in halos.values():
                if hasattr(v, "shape") and len(getattr(v, "shape", ())) >= 1:
                    return int(v.shape[0])

    # Fallback: scan dicts for array-like leaves.
    best = None

    def walk(obj):
        nonlocal best
        if isinstance(obj, dict):
            for v in obj.values():
                walk(v)
        else:
            if hasattr(obj, "shape") and len(getattr(obj, "shape", ())) >= 1:
                n0 = int(obj.shape[0])
                # halo_info files are huge; n0 should be large-ish.
                if best is None or n0 > best:
                    best = n0

    walk(tree)
    if best is None:
        raise RuntimeError("Could not infer halo count from ASDF tree.")
    return best


def count_halos_by_asdf_scan(halo_info_dir: Path) -> tuple[int, list[tuple[str, int]]]:
    import asdf

    files = sorted(halo_info_dir.glob("halo_info_*.asdf"))
    if not files:
        raise FileNotFoundError(f"No halo_info_*.asdf files found in {halo_info_dir}")

    total = 0
    per = []
    for f in files:
        with asdf.open(f, lazy_load=True) as af:
            n = _pick_halo_table_len(af.tree)
        per.append((f.name, n))
        total += n
        print(f"{f.name}: {n:,}")
    return total, per


# Run Option B (requires ASDF to successfully open these files).
try:
    total_halos, per_file = count_halos_by_asdf_scan(HALO_INFO_DIR)
    print(f"\nTotal halos across files: {total_halos:,}")
except Exception as e:
    print("Option B failed (likely missing codec support).")
    print(type(e).__name__, e)
    print("\nRecommended fix: install `abacusutils` into this kernel environment.")